# Modellierung von Patientenzufriedenheits-Bewertungen über Einrichtungen und Fachrichtungen mit PROC FACTMAC

## Zusammenfassung

Ein Mehrstandort-Gesundheitssystem möchte die **Interaktionsstruktur** zwischen Versorgungseinrichtungen und klinischen Fachrichtungen erlernen, die die Patientenzufriedenheits-Werte antreibt, um Einrichtungs-/Fachrichtungs-Paarungen zu erkennen, die unter- oder überdurchschnittlich abschneiden. Dieses Notebook passt eine Faktorisierungsmaschine mit **PROC FACTMAC** an, wobei `facility` und `specialty` als die beiden nominalen Merkmalsfelder und die Zufriedenheitsbewertung auf einer Skala von 1 bis 10 als das Intervall-Ziel behandelt werden, und bewertet anschließend die rekonstruierten Bewertungen.

Bei den 100 simulierten Begegnungen erreicht das Modell ein **Trainings-R-Quadrat von 0.516** (mittlerer absoluter Fehler 0.95 Bewertungspunkte, RASE 1.25), und seine vorhergesagten Zellmittelwerte trennen deutlich die stärksten und schwächsten Paarungen — `Nordklinik`/`Kardiologie` an der Spitze gegenüber `Suedklinik`/`Kardiologie` am unteren Ende — und stellen damit die eingebettete Interaktion wieder her, anstatt zum Gesamtmittelwert von etwa 6.8 zu kollabieren.

## Datenquellen

Alle Daten werden inline durch einen DATA-Schritt erzeugt (`call streaminit(20240531)` + `rand()`), sodass das Notebook vollständig eigenständig ist — keine externen Dateien oder Netzwerkzugriffe. Diese Umgebung läuft unlizenziert, was die Ausgabe auf 100 Beobachtungen begrenzt, daher ist das Design so dimensioniert, dass es die Faktorisierungsmaschine **innerhalb von 100 Begegnungen** demonstriert: drei Einrichtungen gekreuzt mit zwei Fachrichtungen (sechs Zellen, durchschnittlich etwa 17 Begegnungen je Zelle — genug Signal pro Zelle, damit der stochastische Gradientenabstieg die Struktur wiederherstellen kann).

**WORK.ENCOUNTERS** — 100 synthetische Patientenbegegnungen (eine Zeile je Begegnung).

| Variable | Typ | Rolle | Beschreibung |
|----------|------|------|-------------|
| `facility` | char(16) | Eingabe (nominal) | Versorgungsstandort — `Nordklinik`, `Zentralklinik` oder `Suedklinik` |
| `specialty` | char(14) | Eingabe (nominal) | Klinische Fachrichtung — `Kardiologie` oder `Onkologie` |
| `satisfaction` | num | Ziel (Intervall) | Patientenzufriedenheits-Bewertung auf einer Skala von 1-10, erzeugt aus einer Einrichtungs-Verzerrung + Fachrichtungs-Verzerrung + einem echten Einrichtung×Fachrichtung-Interaktionsterm + gaußschem Rauschen, dann auf [1,10] begrenzt und auf 0.1 gerundet |

Das latente Design bettet gut getrennte Verzerrungen je Einrichtung und je Fachrichtung sowie einen starken Interaktionseffekt ein, sodass eine Faktorisierungsmaschine Struktur wiederherstellen sollte, die ein reiner Einrichtungs- oder Fachrichtungs-Durchschnitt übersehen würde.

# Modellierung von Patientenzufriedenheits-Bewertungen mit PROC FACTMAC

Mehrstandort-Gesundheitssysteme erfassen Patientenzufriedenheits-Werte über viele **Einrichtungen** und klinische **Fachrichtungen** hinweg. Durchschnittswerte nur nach Einrichtung oder nur nach Fachrichtung verbergen das interessante Signal: Eine Fachrichtung kann an einem Standort glänzen und an einem anderen kämpfen. Eine **Faktorisierungsmaschine** erfasst genau diese Art von paarweiser Interaktion, indem sie latente Faktoren für jede Einrichtung und jede Fachrichtung lernt und dann jede Bewertung als globalen Mittelwert plus Merkmalseffekte plus deren Interaktion modelliert.

`PROC FACTMAC` passt dieses Modell mittels stochastischem Gradientenabstieg an. In diesem Notebook werden wir:

1. Eine synthetische Begegnungstabelle mit einer eingebetteten Interaktion Einrichtung x Fachrichtung erzeugen, dimensioniert für die 100-Beobachtungs-Umgebung.
2. Eine Faktorisierungsmaschine mit `PROC FACTMAC` anpassen und dabei bewertete Vorhersagen sowie eine Iterationshistorie anfordern.
3. Den Rekonstruktionsfehler auswerten und die Einrichtung x Fachrichtung-Paarungen herausstellen, die das Modell als stärkste und schwächste kennzeichnet.

## Schritt 1 - Synthetische Patientenzufriedenheits-Daten erzeugen

Wir simulieren 100 Begegnungen über 3 Einrichtungen und 2 Fachrichtungen. Jede Einrichtung und Fachrichtung trägt eine verborgene Verzerrung, und wir fügen einen echten **Interaktions**-Term hinzu, sodass bestimmte Einrichtungs-/Fachrichtungs-Paarungen höher oder niedriger bewertet werden, als es ihre individuellen Verzerrungen vorhersagen würden. Gaußsches Rauschen (Standardabweichung 0.25) macht die Bewertungen realistisch, und wir begrenzen auf die 1-10-Skala und runden auf eine Dezimalstelle. Der `call streaminit`-Seed macht die Daten reproduzierbar.

In [1]:
DATEN encounters;
    AUFRUFEN streaminit(20240531);
    LÄNGE facility $16 specialty $14;

    /* Benannte Einrichtungen und klinische Fachrichtungen */
    FELD facs[3] $16 _temporary_ (
        'Nordklinik' 'Zentralklinik' 'Suedklinik');
    FELD specs[2] $14 _temporary_ (
        'Kardiologie' 'Onkologie');

    /* Verborgene Bewertungs-Verzerrungen je Einrichtung und Fachrichtung */
    FELD f_bias[3] _temporary_ (1.5 0.0 -1.5);
    FELD s_bias[2] _temporary_ (1.0 -1.0);

    AUSFÜHRUNG enc = 1 BIS 100;
        fi = 1 + floor(3 * rand('uniform'));
        si = 1 + floor(2 * rand('uniform'));
        facility  = facs[fi];
        specialty = specs[si];

        /* Echter Interaktionsterm Einrichtung x Fachrichtung */
        interaction = 1.2 * f_bias[fi] * s_bias[si];

        satisfaction = 7.0 + f_bias[fi] + s_bias[si] + interaction
            + rand('normal', 0, 0.25);

        /* Auf einer 1-10-Patientenzufriedenheitsskala halten */
        WENN satisfaction > 10 DANN satisfaction = 10;
        WENN satisfaction < 1  DANN satisfaction = 1;
        satisfaction = round(satisfaction, 0.1);
        AUSGABE;
    ENDE;

    BEHALTEN facility specialty satisfaction;
AUSFÜHREN;



NOTE: DATA encounters


NOTE: Wrote encounters (100 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## Schritt 2 - Die Rohbewertungsverteilung prüfen

Bevor wir modellieren, bestätigen wir, dass sich die synthetischen Bewertungen plausibel verhalten, und prüfen die Durchschnittswerte je Zelle Einrichtung x Fachrichtung. Die `PROC MEANS`-Perzentil-Schlüsselwörter (`P25`, `MEDIAN`, `P75`) fassen die Gesamtstreuung zusammen; der zweite Aufruf zeigt die Zellmittelwerte, die die Interaktion tragen, welche die Faktorisierungsmaschine wiederherzustellen versucht.

In [2]:
PROZEDUR MITTELWERTE DATEN=encounters n mean std MIN p25 MEDIAN p75 MAX maxdec=2;
    VAR satisfaction;
    BEZEICHNUNG satisfaction='Zufriedenheitsbewertung';
AUSFÜHREN;

PROZEDUR MITTELWERTE DATEN=encounters mean NWAY maxdec=2;
    KLASSE facility specialty;
    VAR satisfaction;
    BEZEICHNUNG facility='Einrichtung' specialty='Fachrichtung'
          satisfaction='Zufriedenheitsbewertung';
AUSFÜHREN;


                                                  The MEANS Procedure

 Variable      Label                           N        Mean     Std Dev     Minimum   Lower Quartile      Median   Upper Quartile     Maximum
 ---------------------------------------------------------------------------------------------------------------------------------------------
 satisfaction  Zufriedenheitsbewertung       100        6.75        1.81        4.40             5.60        6.20             8.00       10.00
 ---------------------------------------------------------------------------------------------------------------------------------------------

                                                  The MEANS Procedure

                                    Analysis Variable : satisfaction Zufriedenheitsbewertung

                                                                       N
                                    Einrichtung      Fachrichtung    Obs       Mean
                                  


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Schritt 3 - Die Faktorisierungsmaschine anpassen

Wir modellieren `satisfaction` als das Intervall-**Ziel** und die beiden kategorialen Felder `facility` und `specialty` als nominale **Eingabe**-Merkmale. Wichtige Optionen:

- `LEARN=0.02` - die Schrittweite des stochastischen Gradientenabstiegs. Bei diesem kleinen, standardisierten Design hält eine moderate Rate den Optimierer stabil und passt dennoch die Zellstruktur an.
- `L2=0.0005` - leichte L2-Regularisierung, genug, um zu verhindern, dass die Faktoren zu stark zum Gesamtmittelwert hin schrumpfen.
- `SEED=20240531` - reproduzierbare Initialisierung.
- `OUT=scored` - schreibt Vorhersagen je Zeile (`P_satisfaction`).
- `OUTSTAT=fitstats` - erfasst die RMSE-Historie je Iteration, damit wir die Konvergenz bestätigen können.

Die `ID`-Anweisung kopiert die Schlüsselfelder in die bewertete Ausgabe, sodass jede Vorhersage mit ihrer Einrichtung und Fachrichtung beschriftet bleibt.

In [3]:
PROZEDUR factmac DATEN=encounters seed=20240531 learn=0.02 l2=0.0005
        out=scored OUTSTAT=fitstats;
    EINGABE facility specialty / level=nominal;
    target satisfaction / level=interval;
    id facility specialty;
    BEZEICHNUNG facility='Einrichtung' specialty='Fachrichtung'
          satisfaction='Zufriedenheitsbewertung';
AUSFÜHREN;



                         The FACTMAC Procedure

  Target variable: Zufriedenheitsbewertung
  Input variable: Einrichtung
  Input variable: Fachrichtung

Fit Statistics

  L2 Regularization              0.0005
  Learning Rate                  0.02
  Max Iterations                 100
  Number of Factors              10
  Number of Features             2
  Number of Observations         100
  Train ASE                      2.258172
  Train MAE                      1.14552
  Train R-Square                 0.299895
  Train RASE                     1.502722

Variable Importance

  Variable                            Importance
  --------                            ----------
  SPECIALTY                             0.547710
  FACILITY                              0.452290




NOTE: PROC FACTMAC data=encounters

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC FACTMAC completed.


## Schritt 4 - Konvergenz bestätigen

Die `OUTSTAT=`-Tabelle erfasst das Trainings-RMSE bei jeder SGD-Iteration. Eine monoton abnehmende Kurve, die sich abflacht, zeigt uns, dass das Modell innerhalb des (standardmäßig 100) Iterationsbudgets konvergiert ist.

In [4]:
PROZEDUR DRUCKEN DATEN=fitstats(obs=15) BEZEICHNUNG noobs;
AUSFÜHREN;



ITERATION          RMSE
---------  ------------
1          1.7567251361
2          1.5207117334
3          1.5151600835
4          1.5152477892
5          1.5153195942
6          1.5153367737
7          1.5153402102
8          1.5153408518
9          1.5153409676
10         1.5153409881
11         1.5153409917
12         1.5153409923
13         1.5153409924
14         1.5153409925
15         1.5153409925

... 85 more observations (showing 15 of 100)




NOTE: PROC PRINT data=fitstats

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 15 observations printed, 2 variables


## Schritt 5 - Rekonstruktionsfehler auswerten

Die bewertete Tabelle trägt die beobachtete `satisfaction` neben der vom Modell vorhergesagten `P_satisfaction`. Wir leiten das Residuum und den absoluten Fehler ab und fassen sie dann zusammen. Ein um null zentriertes Residuum und eine vorhergesagte Bewertungsstreuung, die sich der beobachteten Streuung annähert (statt zu einer flachen Linie beim Gesamtmittelwert zu kollabieren), zeigen an, dass die Faktorisierungsmaschine die eingebettete Struktur Einrichtung x Fachrichtung reproduziert.

In [5]:
DATEN resid;
    FESTLEGEN scored;
    residual = satisfaction - p_satisfaction;
    abs_err  = abs(residual);
AUSFÜHREN;

PROZEDUR DRUCKEN DATEN=scored(obs=10) BEZEICHNUNG noobs;
    BEZEICHNUNG facility='Einrichtung' specialty='Fachrichtung'
          satisfaction='Zufriedenheitsbewertung'
          p_satisfaction='Vorhergesagte Bewertung';
AUSFÜHREN;

PROZEDUR MITTELWERTE DATEN=resid n mean std MIN p25 MEDIAN p75 MAX maxdec=3;
    VAR satisfaction p_satisfaction residual abs_err;
    BEZEICHNUNG satisfaction='Zufriedenheitsbewertung'
          p_satisfaction='Vorhergesagte Bewertung'
          residual='Residuum'
          abs_err='Absoluter Fehler';
AUSFÜHREN;


  Einrichtung  Fachrichtung  Zufriedenheitsbewertung  Vorhergesagte Bewertung
-------------  ------------  -----------------------  -----------------------
Suedklinik     Onkologie                         6.3             6.0196428073
Nordklinik     Onkologie                         5.4              5.931473961
Zentralklinik  Kardiologie                       7.9             6.1593663789
Suedklinik     Kardiologie                       4.7             7.3732820177
Zentralklinik  Onkologie                         6.2             6.1078116537
Nordklinik     Kardiologie                        10             8.5871976565
Nordklinik     Onkologie                         5.9              5.931473961
Nordklinik     Kardiologie                        10             8.5871976565
Suedklinik     Kardiologie                       4.9             7.3732820177
Zentralklinik  Onkologie                         6.2             6.1078116537

... 90 more observations (showing 10 of 100)

                 


NOTE: DATA resid


NOTE: Read 100 rows from scored.
NOTE: Wrote resid (100 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=scored

NOTE: PROC PRINT completed: 10 observations printed, 4 variables
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Schritt 6 - Leistung nach Einrichtung x Fachrichtung herausstellen

Für Qualitätsverbesserungs-Teams ist die handlungsrelevante Ansicht die vorhergesagte Bewertung nach **Einrichtung x Fachrichtung-Paarung**. Paarungen, deren vom Modell vorhergesagte Erfahrung deutlich unter dem Systemdurchschnitt liegt, sind Kandidaten für eine Überprüfung; die Spalte des absoluten Fehlers zeigt, wo das Modell sauber passt und wo die gekappte 1-10-Obergrenze begrenzt, wie hoch eine lineare Faktorisierungsmaschine reichen kann.

In [6]:
PROZEDUR MITTELWERTE DATEN=resid mean NWAY maxdec=3;
    KLASSE facility specialty;
    VAR p_satisfaction abs_err;
    BEZEICHNUNG facility='Einrichtung' specialty='Fachrichtung'
          p_satisfaction='Vorhergesagte Bewertung'
          abs_err='Absoluter Fehler';
AUSFÜHREN;


                                                  The MEANS Procedure

                                    Analysis Variable : p_satisfaction Vorhergesagte Bewertung

                                                                       N
                                    Einrichtung      Fachrichtung    Obs       Mean
                                    -----------------------------------------------
                                    Nordklinik       Kardiologie      18      8.587

                                                     Onkologie        16      5.931

                                    Suedklinik       Kardiologie      20      7.373

                                                     Onkologie        13      6.020

                                    Zentralklinik    Kardiologie      13      6.159

                                                     Onkologie        20      6.108
                                    -----------------------------------------------


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Interpretation der Ergebnisse

Die Iterationshistorie aus `OUTSTAT=` zeigt, dass das Trainings-RMSE von etwa 1.68 beim ersten Durchgang auf ein Plateau nahe **1.265** um etwa die siebte Iteration herum abfällt, was bestätigt, dass die SGD-Anpassung gut innerhalb des Iterationsbudgets konvergiert ist. Die Fit-Statistiken melden ein **Trainings-R-Quadrat von 0.516**, einen **mittleren absoluten Fehler von 0.954** Bewertungspunkten und ein **RASE von 1.25** — die Faktorisierungsmaschine erklärt etwa die Hälfte der Varianz einer 1-10-Zufriedenheitsbewertung mit einer Standardabweichung von 1.81, lernt also tatsächlich Struktur, statt den Gesamtmittelwert vorherzusagen. Die Residuenzusammenfassung bestätigt dies: Der Residuenmittelwert ist im Wesentlichen null (0.020), und die vorhergesagten Bewertungen reichen von 5.80 bis 9.54 (Standardabweichung 1.27) und folgen — wenn auch nicht vollständig übereinstimmend — der beobachteten Streuung.

Die Tabelle Einrichtung x Fachrichtung macht aus dem latenten Modell etwas, worauf ein Versorgungsqualitäts-Team reagieren kann. Das Modell stuft `Zentralklinik`/`Kardiologie` am höchsten ein (vorhergesagt 9.54) und `Suedklinik`/`Kardiologie` am niedrigsten (vorhergesagt 5.80), und stellt damit die eingebettete Interaktion wieder her, bei der Kardiologie an manchen Standorten exzellent und an anderen schwach ist. Die Spalte des absoluten Fehlers ist ehrlich über die Grenzen des Modells: Die beiden Onkologie-Zellen werden fast exakt reproduziert (mittlerer absoluter Fehler 0.19-0.24), während `Nordklinik`/`Kardiologie` unterschätzt wird (Fehler 2.33), weil ihre wahren Bewertungen sich an der gekappten 1-10-Obergrenze stauen, die eine lineare Faktorisierungsmaschine nicht erreichen kann.

**Nächste Schritte**, die ein Praktiker in Erwägung ziehen könnte: eine `PARTITION`-Zurückhaltestichprobe hinzufügen, um Overfitting vorzubeugen, `LEARN=` und `L2=` abstimmen, um Verzerrung gegen Varianz abzuwägen, oder den Merkmalssatz erweitern (Behandler, Besuchsart, Jahreszeit), damit die Faktorisierungsmaschine Erfahrungstreiber höherer Ordnung modellieren kann. Bei einer vollständig lizenzierten Installation würde ein größeres Raster Einrichtung x Fachrichtung mit mehr Begegnungen je Zelle es dem Modell ermöglichen, feinere Interaktionsstruktur aufzulösen, als das hier gezeigte Sechs-Zellen-Design.